# ChuckleNet WavLM + Prosody Extraction - 555 Videos

## Full Utterance-Level Extraction for Scale-Up

This Colab notebook:
1. Mounts Google Drive to access audio files
2. Downloads audio from Drive (vtt_audio_local.tar.gz)
3. Loads `utterances_clean.jsonl` from GitHub
4. Extracts WavLM (768-dim) + Prosody (21-dim) at **utterance level**
5. Saves combined dataset to Google Drive
6. Trains and validates the model

**Audio:** 621 files (~5GB) - Download from Drive first!
**File ID:** `1jHy11LdU0bv8ra-Lj9xhpKT4yXE3L1Tp`

**Runtime:** ~2-3 hours on T4 GPU

## Step 1: Mount Google Drive

In [ ]:
# Force re-authentication
from google.colab import drive
drive.mount('/content/gdrive', force_auth_changed=True)
print('Drive mounted!')

## Step 2: Install Packages

In [ ]:
!apt-get install -y ffmpeg  # Fix M4A audio loading
!pip install -q transformers librosa scikit-learn

import os
import json
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import librosa
import time
from pathlib import Path

SR = 16000
MAX_DURATION = 5.0
BATCH = 16
EPOCHS = 10
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Step 3: Load Utterances from GitHub

In [ ]:
!wget -q -O /tmp/utterances_clean.jsonl.gz https://github.com/Das-rebel/chuck-audio-notebooks/raw/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utterances_clean.jsonl.gz

utterances = []
with open('/tmp/utterances_clean.jsonl') as f:
    for line in f:
        utterances.append(json.loads(line))

print(f'Loaded {len(utterances)} utterances')

# Build video -> utterances mapping
vtt_data = {}
for u in utterances:
    vid = u['video_id']
    if vid not in vtt_data:
        vtt_data[vid] = []
    vtt_data[vid].append(u)

print(f'Unique videos: {len(vtt_data)}')

# Label distribution
total_laugh = sum(1 for u in utterances if u['label'] == 1)
print(f'Positive rate: {total_laugh}/{len(utterances)} ({total_laugh/len(utterances)*100:.2f}%)')

In [ ]:
# Download audio files from Google Drive
print('Downloading audio files (4.9GB, ~5 min)...')
!pip install -q gdown
!gdown 1jHy11LdU0bv8ra-Lj9xhpKT4yXE3L1Tp -O /tmp/vtt_audio_local.tar.gz
!mkdir -p /content/gdrive/MyDrive/vtt_audio_local
!tar -xzf /tmp/vtt_audio_local.tar.gz -C /content/gdrive/MyDrive/
print('Audio extracted to /content/gdrive/MyDrive/vtt_audio_local/')

## Step 4: Find Audio Files on Google Drive

In [ ]:
# Audio file locations on Google Drive
# Check multiple possible locations
audio_base = Path('/content/gdrive/MyDrive')

# Build audio file map
audio_files = {}
search_folders = [
    audio_base / 'vtt_audio_local',  # PRIMARY: 621 audio files from local backup
    audio_base / 'data' / 'utterances' / 'vtt_audio_local',
    audio_base / 'chuckle_audio',
]

for folder in search_folders:
    if folder.exists():
        print(f'Searching: {folder}')
        for f in folder.glob('*'):
            if f.suffix in ['.m4a', '.mp3', '.wav']:
                vid = f.stem
                audio_files[vid] = str(f)

print(f'\nFound {len(audio_files)} audio files')

# Check coverage
covered = sum(1 for vid in vtt_data if vid in audio_files)
print(f'Utterances with audio: {covered}/{len(vtt_data)} videos')

In [ ]:
# If audio not found in folders, check for tar archive
import os
if len(audio_files) < 100:
    print('Few audio files found. Checking for tar archive...')
    tar_path = audio_base / 'vtt_audio_local.tar.gz'
    if tar_path.exists():
        print(f'Extracting {tar_path}...')
        import subprocess
        subprocess.run(['tar', '-xzf', str(tar_path), '-C', str(audio_base)], check=True)
        print('Extracted! Re-scanning...')
        audio_extracted = audio_base / 'vtt_audio_local'
        if audio_extracted.exists():
            for f in audio_extracted.glob('*'):
                if f.suffix in ['.m4a', '.mp3', '.wav']:
                    vid = f.stem
                    audio_files[vid] = str(f)
            print(f'Found {len(audio_files)} audio files after extraction')


## Step 5: Load WavLM Model

In [ ]:
print('Loading WavLM model...')
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
model = model.to(device).eval()
print('Model loaded!')

## Step 6: Define Extraction Functions

In [ ]:
def extract_prosody(audio_path, start, end, sr=SR):
    """Extract 21-dim prosody features for an audio segment."""
    try:
        y, sr = librosa.load(audio_path, sr=sr, mono=True, offset=start, duration=end-start)
        if len(y) < sr * 0.05:
            return np.zeros(21, dtype=np.float32)
        
        feats = []
        
        # 1. F0/pitch features (5 dims)
        try:
            f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
            f0_valid = f0[~np.isnan(f0)]
            feats.extend([
                np.mean(f0_valid) if len(f0_valid) > 0 else 0,
                np.std(f0_valid) if len(f0_valid) > 0 else 0,
                np.max(f0_valid) if len(f0_valid) > 0 else 0,
                np.min(f0_valid) if len(f0_valid) > 0 else 0,
                np.median(f0_valid) if len(f0_valid) > 0 else 0,
            ])
        except:
            feats.extend([0, 0, 0, 0, 0])
        
        # 2. Energy features (4 dims)
        rms = librosa.feature.rms(y=y)[0]
        feats.extend([
            np.mean(rms),
            np.std(rms),
            np.max(rms),
            np.min(rms),
        ])
        
        # 3. Spectral features (5 dims)
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        spec_flat = librosa.feature.spectral_flatness(y=y)[0]
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        feats.extend([
            np.mean(spec_cent),
            np.mean(spec_bw),
            np.mean(spec_flat),
            np.mean(zcr),
            np.max(zcr),
        ])
        
        # 4. MFCCs (7 dims)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=7)
        for i in range(7):
            feats.append(np.mean(mfccs[i]))
        
        return np.array(feats, dtype=np.float32)
    except Exception as e:
        return np.zeros(21, dtype=np.float32)

def extract_wavlm(audio_path, start, end):
    """Extract WavLM embedding for an audio segment."""
    try:
        y, sr = librosa.load(audio_path, sr=SR, mono=True, offset=start, duration=end-start)
        if len(y) < SR * 0.05:
            return np.zeros(768, dtype=np.float32)
        
        y_tensor = torch.tensor(y).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(y_tensor)
            embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return embedding
    except:
        return np.zeros(768, dtype=np.float32)

## Step 7: Extract Features

In [ ]:
print('Starting extraction...')
print(f'Processing {len(utterances)} utterances from {len(vtt_data)} videos')

all_embeddings = []
all_prosody = []
all_labels = []
all_uids = []

start_time = time.time()
for i, u in enumerate(utterances):
    vid = u['video_id']
    if vid not in audio_files:
        continue
    
    audio_path = audio_files[vid]
    start = u['start']
    end = u['end']
    label = u['label']
    uid = f"{vid}_{start:.2f}_{end:.2f}"
    
    emb = extract_wavlm(audio_path, start, end)
    pros = extract_prosody(audio_path, start, end)
    
    all_embeddings.append(emb)
    all_prosody.append(pros)
    all_labels.append(label)
    all_uids.append(uid)
    
    if (i + 1) % 1000 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        eta = (len(utterances) - i - 1) / rate / 60
        print(f'Progress: {i+1}/{len(utterances)} ({rate:.1f} utt/s, ETA: {eta:.1f} min)')

print(f'\nExtraction complete! {len(all_embeddings)} utterances processed')
print(f'Total time: {(time.time() - start_time)/60:.1f} minutes')

## Step 8: Convert to Arrays and Save

In [ ]:
embeddings = np.array(all_embeddings, dtype=np.float32)
prosody = np.array(all_prosody, dtype=np.float32)
labels = np.array(all_labels, dtype=np.int64)

print(f'Embeddings shape: {embeddings.shape}')
print(f'Prosody shape: {prosody.shape}')
print(f'Positive rate: {np.sum(labels==1)/len(labels)*100:.2f}%')

# Save to Google Drive
output_dir = Path('/content/gdrive/MyDrive')
np.savez(
    output_dir / 'wavlm_prosody_555_complete.npz',
    embeddings=embeddings,
    prosody=prosody,
    labels=labels,
    uids=np.array(all_uids)
)
print(f'Saved to {output_dir / "wavlm_prosody_555_complete.npz"}')

## Step 9: Train Model (Sanity Check)

In [ ]:
print('Running quick training sanity check...')

# Normalize prosody
prosody_scaler = StandardScaler()
prosody_scaled = prosody_scaler.fit_transform(prosody)

# Combine features
X = np.hstack([embeddings, prosody_scaled])
y = labels

# Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

print(f'Train: {len(X_train)}, Val: {len(X_val)}')
print(f'Positive rate - Train: {np.mean(y_train)*100:.2f}%, Val: {np.mean(y_val)*100:.2f}%')

In [ ]:
# Model class
class WavLMProsodyClassifier(nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=21, hidden_dim=256):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(wavlm_dim + prosody_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        return self.fc(x)

model = WavLMProsodyClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print('Model initialized!')

In [ ]:
# Quick training (5 epochs)
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.long).to(device)

for epoch in range(5):
    model.train()
    losses = []
    for i in range(0, len(X_train_t), BATCH):
        batch_x = X_train_t[i:i+BATCH]
        batch_y = y_train_t[i:i+BATCH]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t).argmax(dim=1).cpu().numpy()
        val_f1 = f1_score(y_val, val_preds)
    
    print(f'Epoch {epoch+1}/5 - Loss: {np.mean(losses):.4f} - Val F1: {val_f1:.4f}')

print('\nTraining complete!')